In [1]:
import os
import cv2
import math
import mediapipe as mp
import pandas as pd
import numpy as np
from collections import defaultdict

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (d:\Fontys\Semester 4 - ML\Sign Language Recognition\Project\venv\Lib\site-packages\google\protobuf\__init__.py)

In [ ]:
TRAIN_DIR = "../data/alphabet/raw/train"
TEST_DIR = "../data/alphabet/raw/test"

OUTPUT_TRAIN_CSV = "../data/alphabet/landmarks/train_landmarks_normalized.csv"
OUTPUT_TEST_CSV = "../data/alphabet/landmarks/test_landmarks_normalized.csv"

In [ ]:
mp_hands = mp.solutions.hands

In [ ]:
def preprocess_image_for_mediapipe(image, pad=80, target_size=512):
    if image is None:
        return None

    image = cv2.copyMakeBorder(
        image,
        pad, pad, pad, pad,
        borderType=cv2.BORDER_CONSTANT,
        value=(255, 255, 255)
    )

    image = cv2.resize(image, (target_size, target_size))
    return image

In [ ]:
def normalize_landmarks(landmarks):
    wrist = landmarks[0]

    shifted = []
    for x, y, z in landmarks:
        shifted.append((x - wrist[0], y - wrist[1], z - wrist[2]))

    max_dist = 0.0
    for x, y, z in shifted:
        dist = math.sqrt(x**2 + y**2 + z**2)
        if dist > max_dist:
            max_dist = dist

    if max_dist == 0:
        return None

    normalized = []
    for x, y, z in shifted:
        normalized.extend([x / max_dist, y / max_dist, z / max_dist])

    return normalized

In [ ]:
def extract_landmarks_from_image(image_path, hands):
    image = cv2.imread(image_path)
    if image is None:
        return None

    candidates = []

    # original
    candidates.append(image)

    # smaller padding versions
    for pad in [0, 10, 20, 40]:
        padded = cv2.copyMakeBorder(
            image, pad, pad, pad, pad,
            borderType=cv2.BORDER_CONSTANT,
            value=(255, 255, 255)
        )
        candidates.append(padded)

    # flipped versions too
    candidates += [cv2.flip(img, 1) for img in candidates]

    for img in candidates:
        img = cv2.resize(img, (512, 512))
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks:
            coords = [(lm.x, lm.y, lm.z) for lm in results.multi_hand_landmarks[0].landmark]
            normalized = normalize_landmarks(coords)
            if normalized is not None:
                return normalized

    return None

In [ ]:
def process_dataset(input_dir, split_name):
    data = []
    failed_files = []
    stats = defaultdict(lambda: {"total": 0, "kept": 0, "failed": 0})

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.3
    ) as hands:

        for label in sorted(os.listdir(input_dir)):
            label_path = os.path.join(input_dir, label)

            if not os.path.isdir(label_path):
                continue

            print(f"Processing {split_name} label: {label}")

            for file_name in os.listdir(label_path):
                file_path = os.path.join(label_path, file_name)
                stats[label]["total"] += 1

                features = extract_landmarks_from_image(file_path, hands)

                if features is not None:
                    row = features + [label, file_path, split_name]
                    data.append(row)
                    stats[label]["kept"] += 1
                else:
                    stats[label]["failed"] += 1
                    failed_files.append({
                        "label": label,
                        "file_path": file_path,
                        "split": split_name
                    })

    columns = []
    for i in range(21):
        columns.extend([f"x{i}", f"y{i}", f"z{i}"])
    columns += ["label", "file_path", "split"]

    df = pd.DataFrame(data, columns=columns)
    stats_df = pd.DataFrame(stats).T.reset_index().rename(columns={"index": "label"})
    failed_df = pd.DataFrame(failed_files)

    return df, stats_df, failed_df

In [ ]:
train_df, train_stats, train_failed = process_dataset(TRAIN_DIR, "train")
test_df, test_stats, test_failed = process_dataset(TEST_DIR, "test")

AttributeError: 'google._upb._message.FieldDescriptor' object has no attribute 'label'

In [ ]:
os.makedirs("../data/alphabet/landmarks", exist_ok=True)

train_df.to_csv(OUTPUT_TRAIN_CSV, index=False)
test_df.to_csv(OUTPUT_TEST_CSV, index=False)

print("Saved:")
print(OUTPUT_TRAIN_CSV)
print(OUTPUT_TEST_CSV)

Saved:
../data/alphabet/landmarks/train_landmarks_normalized.csv
../data/alphabet/landmarks/test_landmarks_normalized.csv


In [ ]:
train_failed.head(20)

,label,file_path,split
0,A,../data/alphabet/raw/train\A\Image_1685009090....,train
1,A,../data/alphabet/raw/train\A\Image_1685009110....,train
2,A,../data/alphabet/raw/train\A\Image_1685009112....,train
3,A,../data/alphabet/raw/train\A\Image_1685009115....,train
4,A,../data/alphabet/raw/train\A\Image_1685009115....,train
5,A,../data/alphabet/raw/train\A\Image_1685009120....,train
6,A,../data/alphabet/raw/train\A\Image_1685009120....,train
7,A,../data/alphabet/raw/train\A\Image_1685009122....,train
8,A,../data/alphabet/raw/train\A\Image_1685009124....,train
9,A,../data/alphabet/raw/train\A\Image_1685009125....,train


In [ ]:
train_stats["keep_rate"] = train_stats["kept"] / train_stats["total"]
test_stats["keep_rate"] = test_stats["kept"] / test_stats["total"]

train_stats.sort_values("kept", ascending=False)

,label,total,kept,failed,keep_rate
6,G,435,237,198,0.544828
7,H,432,234,198,0.541667
14,P,438,173,265,0.394977
9,K,455,163,292,0.358242
0,A,447,135,312,0.302013
23,Y,438,105,333,0.239726
4,E,441,103,338,0.233560
15,Q,449,77,372,0.171492
18,T,414,74,340,0.178744
8,I,433,74,359,0.170901


In [ ]:
test_stats.sort_values("kept", ascending=False)

,label,total,kept,failed,keep_rate
7,H,75,44,31,0.586667
4,E,75,32,43,0.426667
14,P,75,28,47,0.373333
15,Q,75,26,49,0.346667
10,L,75,18,57,0.240000
9,K,75,15,60,0.200000
5,F,75,10,65,0.133333
0,A,75,7,68,0.093333
6,G,75,5,70,0.066667
3,D,75,3,72,0.040000
